# Pazarlık Ajanları: Gemma 4 ek deneyi

Ana deney gemma3:4b ile yapıldı. Bu notebook aynı deneyi daha büyük ve yeni bir modelle (gemma4:12b) çalıştırır; sonuçlar ayrı dosyaya yazılır (`eval/results_gemma4.jsonl`), ana sonuçlara dokunulmaz.

Ayarlar: **Accelerator: GPU T4 x2**, **Internet: On**. Önce 1-4. hücreleri çalıştırıp kısa denemenin süresine bakın; sonra 5. hücreyi.

In [ ]:
# 1) Ollama'yı kur ve başlat
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
MODEL = "gemma4:12b"   # daha hızlı ama daha küçük: "gemma4:e4b"
!ollama pull {MODEL}

In [ ]:
# 2) Projeyi indir ve bağımlılıkları kur
!git clone -q https://github.com/nurayy12/ajan_gadget.git
%cd ajan_gadget
!pip install -q -r requirements.txt

In [ ]:
# 3) Ortam değişkenleri ve modelin bizim kodla çalıştığını kontrol
import os
os.environ.update(ANTHROPIC_BASE_URL="http://localhost:11434", ANTHROPIC_API_KEY="ollama", MODEL_NAME=MODEL)
os.environ.pop("USE_MOCK_LLM", None)
import anthropic
r = anthropic.Anthropic().messages.create(model=MODEL, max_tokens=80,
    messages=[{"role": "user", "content": "Tek cümleyle Türkçe selam ver."}])
print([b.type for b in r.content])
print("".join(getattr(b, "text", "") for b in r.content if getattr(b, "type", "text") == "text"))
!ollama ps

In [ ]:
# 4) Kısa deneme: 3 senaryo, ayrı sonuç dosyası. Çıkan 'X dk sürdü' x 40 = 3 tekrarlı tam deney süresi
!python -m eval.run_experiments --limit 3 --repeats 1 --yes --out eval/results_gemma4_deneme.jsonl

In [ ]:
# 5) Tam deney (1 tekrar; süre yetiyorsa --repeats 3) ve metrikler
!python -m eval.run_experiments --repeats 1 --yes --out eval/results_gemma4.jsonl
!python -m eval.metrics --results eval/results_gemma4.jsonl --out-dir eval/out/gemma4
!zip -qr /kaggle/working/sonuclar_gemma4.zip eval/results_gemma4.jsonl eval/out/gemma4
!cat eval/out/gemma4/summary.md

## İsteğe bağlı: demoyu bu modelle açmak

Aşağıdaki iki hücre Streamlit arayüzünü başlatır ve geçici bir internet adresi verir. Notebook açık kaldığı sürece çalışır; adres her seferinde değişir.

In [ ]:
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(5)

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8501 2>&1 | grep -m1 -o "https://.*trycloudflare.com"